In [1]:
"""
================================================================================
Lateral Vehicle Dynamics — Lane Change Case Study
HT-MPC (homothetic, proposed) vs RT-MPC (fixed-tube robust)

Both controllers use the SAME disturbance bound D-bar and the SAME RPI
ellipsoid Omega.  The only difference is the scale adaptation:
  HT-MPC:  alpha_k = ||e_k||_{P^{-1}},  OCP uses closed-form sequence
  RT-MPC:  alpha_k = 1 always (full worst-case tightening)

System
------
Lateral bicycle model linearised at v_x0 = 60 km/h.
  States:  x = [e_y (m), e_psi (rad), v_y (m/s), psi_dot (rad/s)]
  Inputs:  u = [delta_s (rad), mu_b (N·m)]
  Output:  y = e_y  (lateral deviation from road centre)
  Ts = 0.1 s  (ZOH)

Uncertainty
-----------
True longitudinal speed:  v_x = v_x0 + Delta_v_x,  Delta_v_x ~ N(0, sigma_vx^2)
First-order Taylor:       Delta_A = Delta_v_x * D,  D = dA_d/dv_x|_{v_x0}
  => vec(Delta_A) ~ N(0, Sigma_DA),  df_DA = 1  (one scalar source)
  => Sigma_DA = sigma_vx^2 * vec(D) vec(D)^T + eps*I
Input matrix B is independent of v_x  =>  Delta_B = 0 (small eps regularisation).

Shared bound
-----------
Both controllers are designed with the same confidence level p = 0.99,
giving a single shared Omega and D-bar.  The comparison isolates the
effect of the homothetic scale adaptation, not of different bound sizes.

Lane change scenario
--------------------
Slow lane: e_y = -1.75 m,  fast lane: e_y = +1.75 m.
Step reference change at step T_lc = 12  (t = 1.2 s).
Simulation length: T = 50 steps  (5 s).

Figure produced
---------------
fig_lane_change_comparison_alt.png
  Top: y [m] vs x [m] — spatial trajectory with projected tube band
================================================================================
"""

import numpy as np
import scipy.linalg  as la
import scipy.signal  as sig
import scipy.stats   as stats
import cvxpy         as cp
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.patches import Patch
from matplotlib.lines   import Line2D
from dataclasses import dataclass
from itertools   import product as iprod

# ── matplotlib style (IEEE single-column) ─────────────────────────────────────
FP      = 8
ONE_COL = 3.5

plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":         FP,    "axes.labelsize":   FP,
    "axes.titlesize":    FP,    "xtick.labelsize":  FP - 1,
    "ytick.labelsize":   FP - 1,"legend.fontsize":  FP - 1,
    "legend.framealpha": 0.90,  "legend.frameon":   True,
    "legend.edgecolor":  "0.80","lines.linewidth":  1.0,
    "axes.linewidth":    0.6,   "grid.linewidth":   0.4,
    "grid.alpha":        0.40,  "savefig.dpi":      300,
    "savefig.bbox":      "tight","pdf.fonttype":    42,
    "ps.fonttype":       42,
})

C = {
    "p":      "#007E5C",     # HT-MPC  — green
    "r":      "#FF4500",     # RT-MPC  — orange-red
    "ref":    "#1D1D1DCA",   # reference — near-black
    "constr": "#A80000",     # lane boundaries — dark red
}
LABELS = {"p": "HT-MPC", "r": "RT-MPC"}


# ══════════════════════════════════════════════════════════════════════════════
#  VEHICLE MODEL
# ══════════════════════════════════════════════════════════════════════════════

_m  = 1500.0;  _Iz = 2500.0
_lf = 1.2;     _lr = 1.6
_Cf = 60000.0; _Cr = 70000.0


def _Ac(vx):
    m, Iz, lf, lr, Cf, Cr = _m, _Iz, _lf, _lr, _Cf, _Cr
    return np.array([
        [0.,  vx,  1.,  0.],
        [0.,  0.,  0.,  1.],
        [0.,  0.,  -2.*(Cf+Cr)/(m*vx),
                   (-2.*(Cf*lf-Cr*lr)/(m*vx) - vx)],
        [0.,  0.,  -2.*(Cf*lf-Cr*lr)/(Iz*vx),
                   -2.*(Cf*lf**2+Cr*lr**2)/(Iz*vx)],
    ])


def _Bc():
    m, Iz, lf, Cf = _m, _Iz, _lf, _Cf
    return np.array([
        [0.,           0.   ],
        [0.,           0.   ],
        [2.*Cf/m,      0.   ],
        [2.*Cf*lf/Iz,  1./Iz],
    ])


def discretize(vx, Ts):
    sd = sig.StateSpace(
        _Ac(vx), _Bc(), np.eye(4), np.zeros((4, 2))
    ).to_discrete(Ts)
    return np.array(sd.A), np.array(sd.B)


def dAd_dvx(vx0, Ts, h=1e-4):
    """Numerical Jacobian D = dA_d/dv_x at v_x0."""
    Ap, _ = discretize(vx0 + h, Ts)
    Am, _ = discretize(vx0 - h, Ts)
    return (Ap - Am) / (2.*h)


# ══════════════════════════════════════════════════════════════════════════════
#  UNCERTAINTY MODEL  (shared by both controllers)
# ══════════════════════════════════════════════════════════════════════════════

class UncertaintyModel:
    """
    Gaussian confidence-region instantiation of the bounded sets A, B, W.
    df_DA = 1 because Delta_A = Delta_vx * D is driven by ONE scalar.
    """
    def __init__(self, Sigma_DA, Sigma_DB, Sigma_w,
                 p_A, p_B, p_W, n, m, df_DA=1, df_DB=1, label=""):
        self.Sigma_DA = np.atleast_2d(Sigma_DA)
        self.Sigma_DB = np.atleast_2d(Sigma_DB)
        self.Sigma_w  = np.atleast_2d(Sigma_w)
        self.p_A = p_A;  self.p_B = p_B;  self.p_W = p_W
        self.n = n;  self.m = m
        self.chi2_DA = float(stats.chi2.ppf(p_A, df=df_DA))
        self.chi2_DB = float(stats.chi2.ppf(p_B, df=df_DB))
        self.chi2_w  = float(stats.chi2.ppf(p_W, df=n))
        tag = f" [{label}]" if label else ""
        print(f"  OK{tag}  p_A={p_A}, p_B={p_B}, p_W={p_W}  "
              f"chi2_DA={self.chi2_DA:.3f}  chi2_w={self.chi2_w:.3f}")

    def QA_at_x(self, x):
        L = np.kron(np.eye(self.n), x.reshape(1, -1))
        Q = L @ self.Sigma_DA @ L.T
        return 0.5*(Q + Q.T)

    def QB_at_u(self, u):
        L = np.kron(np.eye(self.n), u.reshape(1, -1))
        Q = L @ self.Sigma_DB @ L.T
        return 0.5*(Q + Q.T)


# ══════════════════════════════════════════════════════════════════════════════
#  VERTEX BOUND (Lemma 1 of the paper)
# ══════════════════════════════════════════════════════════════════════════════

def _box_vertices(H, h):
    d    = H.shape[1];  half = len(h)//2
    assert H.shape[0] == 2*d
    assert np.allclose(H, np.vstack([np.eye(d), -np.eye(d)]))
    return np.array(list(iprod(*zip(-h[half:], h[:d]))))


def compute_Qmax(unc, Hx, h_x, Hu, h_u):
    n = unc.n
    def _sdp(mats):
        Qv = cp.Variable((n, n), symmetric=True)
        cp.Problem(
            cp.Minimize(cp.trace(Qv)),
            [Qv >> M + 1e-9*np.eye(n) for M in mats]
        ).solve(solver="SCS", verbose=False, eps=1e-8, max_iters=15_000)
        if Qv.value is None:
            return mats[0] + 1e-6*np.eye(n)
        return 0.5*(Qv.value + Qv.value.T) + 1e-9*np.eye(n)
    QA_max = _sdp([unc.QA_at_x(v) for v in _box_vertices(Hx, h_x)])
    QB_max = _sdp([unc.QB_at_u(v) for v in _box_vertices(Hu, h_u)])
    print(f"  OK  tr(Q^A_max)={np.trace(QA_max):.5f}  "
          f"tr(Q^B_max)={np.trace(QB_max):.5f}")
    return QA_max, QB_max


# ══════════════════════════════════════════════════════════════════════════════
#  RPI ELLIPSOID  (Prop. 1 of the paper — Stein closed form)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class RPIData:
    P_star:     np.ndarray
    P_star_inv: np.ndarray
    c_S:        float
    lam_star:   float
    S_max:      np.ndarray


def compute_rpi(A_K, unc, QA_max, QB_max, label=""):
    a = [unc.chi2_DA*float(np.trace(QA_max)),
         unc.chi2_DB*float(np.trace(QB_max)),
         unc.chi2_w *float(np.trace(unc.Sigma_w))]
    s = sum(np.sqrt(ai) for ai in a)
    mu = np.array([np.sqrt(ai)/s for ai in a])
    S_max = ((unc.chi2_DA/mu[0])*QA_max
           + (unc.chi2_DB/mu[1])*QB_max
           + (unc.chi2_w /mu[2])*unc.Sigma_w)
    S_max = 0.5*(S_max + S_max.T)

    def _stein(lam):
        P = la.solve_discrete_lyapunov(A_K/np.sqrt(lam), S_max/(1.-lam))
        return 0.5*(P + P.T)

    rho2 = float(np.max(np.abs(la.eigvals(A_K))))**2
    a_, b_ = rho2+1e-8, 1.-1e-8
    gr = (np.sqrt(5.)-1.)/2.
    x1 = b_-gr*(b_-a_);  f1 = np.trace(_stein(x1))
    x2 = a_+gr*(b_-a_);  f2 = np.trace(_stein(x2))
    for _ in range(400):
        if b_-a_ < 1e-12: break
        if f1 < f2:
            b_,x2,f2 = x2,x1,f1; x1=b_-gr*(b_-a_); f1=np.trace(_stein(x1))
        else:
            a_,x1,f1 = x1,x2,f2; x2=a_+gr*(b_-a_); f2=np.trace(_stein(x2))
    lam_star = 0.5*(a_+b_)
    P_star   = _stein(lam_star)
    Pnh      = np.real(la.inv(la.sqrtm(P_star)))
    raw_cS   = (float(np.max(np.real(la.eigvals(Pnh@S_max@Pnh))))
                /(1.-lam_star))
    c_S = float(np.clip(raw_cS, 1e-12, 1.))
    tag = f" [{label}]" if label else ""
    print(f"  OK{tag}  lam*={lam_star:.4f}  c_S={c_S:.6f}  "
          f"tr(P*)={np.trace(P_star):.4f}  "
          f"sqrt(P*[0,0])={np.sqrt(P_star[0,0]):.4f} m")
    return RPIData(P_star=P_star, P_star_inv=la.inv(P_star),
                   c_S=c_S, lam_star=lam_star, S_max=S_max)


# ══════════════════════════════════════════════════════════════════════════════
#  TIGHTENING + TERMINAL SET
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class TightData:
    s_x:    np.ndarray
    s_u:    np.ndarray
    gamma:  float
    L_inf:  object
    sq_gam: float


def compute_tightening(A_K, K, rpi, Hx, h_x, Hu, h_u, P_inf, label=""):
    P = rpi.P_star;  Pi = la.inv(P_inf);  KPK = K@P@K.T
    s_x = np.array([np.sqrt(max(float(h@P@h),   0.)) for h in Hx])
    s_u = np.array([np.sqrt(max(float(h@KPK@h), 0.)) for h in Hu])
    gammas = []
    for i, h in enumerate(Hx):
        mg = h_x[i]-s_x[i];  sv = float(h@Pi@h)
        if mg > 1e-10 and sv > 1e-12: gammas.append(mg**2/sv)
    for i, h in enumerate(Hu):
        hz = K.T@h;  mg = h_u[i]-s_u[i];  sv = float(hz@Pi@hz)
        if mg > 1e-10 and sv > 1e-12: gammas.append(mg**2/sv)
    gamma  = float(np.min(gammas)) if gammas else 0.
    L_inf  = np.linalg.cholesky(P_inf).T if gamma > 1e-8 else None
    sq_gam = float(np.sqrt(gamma))       if gamma > 1e-8 else 0.
    tag = f" [{label}]" if label else ""
    print(f"  OK{tag}  s_x={np.round(s_x,4)}  gamma={gamma:.5f}  "
          f"margins: {np.round(h_x-s_x,4)}")
    return TightData(s_x=s_x, s_u=s_u, gamma=gamma,
                     L_inf=L_inf, sq_gam=sq_gam)


# ══════════════════════════════════════════════════════════════════════════════
#  CLOSED-FORM ALPHA SEQUENCE  (Prop. 3 of the paper)
# ══════════════════════════════════════════════════════════════════════════════

def alpha_seq(alpha_k, c_S, N):
    j = np.arange(N+1, dtype=float)
    return np.sqrt(np.maximum(1.-(1.-c_S)**j*(1.-alpha_k**2), 0.))


# ══════════════════════════════════════════════════════════════════════════════
#  STEADY STATE
# ══════════════════════════════════════════════════════════════════════════════

def steady_state(A, B, C_out, r_ref):
    n_, m_ = A.shape[0], B.shape[1];  p_ = C_out.shape[0]
    r = np.atleast_1d(float(r_ref))
    M   = np.block([[A-np.eye(n_), B], [C_out, np.zeros((p_,m_))]])
    rhs = np.concatenate([np.zeros(n_), r])
    sol, *_ = np.linalg.lstsq(M, rhs, rcond=None)
    return sol[:n_], sol[n_:]


# ══════════════════════════════════════════════════════════════════════════════
#  HT-MPC OCP
# ══════════════════════════════════════════════════════════════════════════════

class HomotheticTubeMPC_OCP:
    """
    Proposed method: per-step tightening driven by the precomputed
    closed-form scale sequence {alpha_{j|k}}.  Standard QP, no SOC
    on the tube scale (only on the terminal set).
    """
    def __init__(self, A, B, Hx, h_x, Hu, h_u, tight, N, Q_c, R_c, P_inf):
        print("[Compile] HT-MPC OCP ...")
        self.A=A; self.B=B; self.Hx=Hx; self.h_x=h_x
        self.Hu=Hu; self.h_u=h_u; self.tight=tight; self.N=N
        n=A.shape[0]; m=B.shape[1]
        Q_c=np.atleast_2d(Q_c); R_c=np.atleast_2d(R_c)
        use_t = tight.gamma > 1e-8
        self.z0_p=cp.Parameter(n); self.zr_p=cp.Parameter(n)
        self.vr_p=cp.Parameter(m)
        self.hz_p=[cp.Parameter(Hx.shape[0]) for _ in range(N+1)]
        self.hv_p=[cp.Parameter(Hu.shape[0]) for _ in range(N)]
        z=cp.Variable((N+1,n)); v=cp.Variable((N,m))
        self._z=z; self._v=v
        cost=0.; cons=[z[0]==self.z0_p]
        for j in range(N):
            cost+=(cp.quad_form(z[j]-self.zr_p,Q_c)
                  +cp.quad_form(v[j]-self.vr_p,R_c))
            cons+=[z[j+1]==A@z[j]+B@v[j],
                   Hx@z[j]<=self.hz_p[j],
                   Hu@v[j]<=self.hv_p[j]]
        cost+=cp.quad_form(z[N]-self.zr_p,P_inf)
        cons.append(Hx@z[N]<=self.hz_p[N])
        if use_t:
            cons.append(cp.norm(tight.L_inf@(z[N]-self.zr_p),2)<=tight.sq_gam)
        self.prob=cp.Problem(cp.Minimize(cost),cons)
        self._solver=cp.CLARABEL if use_t else cp.OSQP
        print(f"  OK  N={N}  solver={self._solver}  terminal_SOC={use_t}")

    def solve(self, z_k, z_ref, v_ref, aseq, warm=True):
        self.z0_p.value=z_k; self.zr_p.value=z_ref; self.vr_p.value=v_ref
        for j in range(self.N+1):
            self.hz_p[j].value=self.h_x-aseq[j]*self.tight.s_x
        for j in range(self.N):
            self.hv_p[j].value=self.h_u-aseq[j]*self.tight.s_u
        try:
            kw = dict(warm_start=warm, verbose=False)
            if self._solver==cp.OSQP:
                self.prob.solve(solver=cp.OSQP,eps_abs=1e-7,eps_rel=1e-7,
                                max_iter=40_000,polish=True,**kw)
            else:
                self.prob.solve(solver=cp.CLARABEL,**kw)
            ok=self.prob.status in ("optimal","optimal_inaccurate")
            return (self._z.value.copy(),self._v.value.copy(),True) if ok \
                   else (None,None,False)
        except Exception as e:
            print(f"  HT-MPC OCP error: {e}"); return None,None,False


# ══════════════════════════════════════════════════════════════════════════════
#  RT-MPC OCP  (fixed tube, alpha = 1 always)
# ══════════════════════════════════════════════════════════════════════════════

class FixedTubeMPC_OCP:
    """
    Classical robust tube MPC.  alpha = 1 always -> fixed worst-case
    tightening.  Uses the SAME Omega (and hence same s_x, s_u) as HT-MPC.
    """
    def __init__(self, A, B, Hx, h_x, Hu, h_u, tight, N, Q_c, R_c, P_inf):
        print("[Compile] RT-MPC OCP ...")
        self.N=N; n=A.shape[0]; m=B.shape[1]
        Q_c=np.atleast_2d(Q_c); R_c=np.atleast_2d(R_c)
        use_t=tight.gamma>1e-8
        hz_fix=h_x-tight.s_x; hv_fix=h_u-tight.s_u
        self.z0_p=cp.Parameter(n); self.zr_p=cp.Parameter(n)
        self.vr_p=cp.Parameter(m)
        z=cp.Variable((N+1,n)); v=cp.Variable((N,m))
        self._z=z; self._v=v
        cost=0.; cons=[z[0]==self.z0_p]
        for j in range(N):
            cost+=(cp.quad_form(z[j]-self.zr_p,Q_c)
                  +cp.quad_form(v[j]-self.vr_p,R_c))
            cons+=[z[j+1]==A@z[j]+B@v[j],
                   Hx@z[j]<=hz_fix,
                   Hu@v[j]<=hv_fix]
        cost+=cp.quad_form(z[N]-self.zr_p,P_inf)
        cons.append(Hx@z[N]<=hz_fix)
        if use_t:
            cons.append(cp.norm(tight.L_inf@(z[N]-self.zr_p),2)<=tight.sq_gam)
        self.prob=cp.Problem(cp.Minimize(cost),cons)
        self._solver=cp.CLARABEL if use_t else cp.OSQP
        print(f"  OK  fixed tightening h_x-s_x={np.round(hz_fix[:4],4)}")

    def solve(self, z_k, z_ref, v_ref, warm=True):
        self.z0_p.value=z_k; self.zr_p.value=z_ref; self.vr_p.value=v_ref
        try:
            kw=dict(warm_start=warm,verbose=False)
            if self._solver==cp.OSQP:
                self.prob.solve(solver=cp.OSQP,eps_abs=1e-7,eps_rel=1e-7,
                                max_iter=40_000,polish=True,**kw)
            else:
                self.prob.solve(solver=cp.CLARABEL,**kw)
            ok=self.prob.status in ("optimal","optimal_inaccurate")
            return (self._z.value.copy(),self._v.value.copy(),True) if ok \
                   else (None,None,False)
        except Exception as e:
            print(f"  RT-MPC OCP error: {e}"); return None,None,False


# ══════════════════════════════════════════════════════════════════════════════
#  SIMULATION
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class SimResult:
    x:      np.ndarray
    z:      np.ndarray
    u:      np.ndarray
    alpha:  np.ndarray
    y_ref:  np.ndarray
    viol_x: np.ndarray
    viol_u: np.ndarray
    n_viol: int
    ok:     bool


def run_htmpc(A, B, A_hat, B_hat, K, C_out,
              Hx, h_x, Hu, h_u, Sigma_w,
              rpi, ocp, y_ref_sig, x0, N, rng_seed=None):
    rng=np.random.RandomState(rng_seed)
    T=len(y_ref_sig); n=A.shape[0]; m=B.shape[1]
    Xt=np.zeros((T,n)); Zt=np.zeros((T,n)); Ut=np.zeros((T,m))
    At=np.zeros(T); vx=np.zeros(T,bool); vu=np.zeros(T,bool)
    ok=True; n_viol=0
    x_k=x0.copy(); z_k=x0.copy()
    for k in range(T):
        zr_k,vr_k=steady_state(A,B,C_out,float(y_ref_sig[k]))
        e_k=x_k-z_k
        raw=float(np.sqrt(max(float(e_k@rpi.P_star_inv@e_k),0.)))
        if raw>1.+1e-6: n_viol+=1
        At[k]=raw; alpha_k=min(raw,1.)
        aseq=alpha_seq(alpha_k,rpi.c_S,N)
        z_opt,v_opt,feas=ocp.solve(z_k,zr_k,vr_k,aseq)
        if not feas: ok=False; break
        u_k=v_opt[0]+K@e_k
        Xt[k]=x_k; Zt[k]=z_k; Ut[k]=u_k
        vx[k]=bool(np.any(Hx@x_k>h_x+1e-6))
        vu[k]=bool(np.any(Hu@u_k>h_u+1e-6))
        w_k=rng.multivariate_normal(np.zeros(n),Sigma_w)
        x_k=A_hat@x_k+B_hat@u_k+w_k
        z_k=A@Zt[k]+B@v_opt[0]
    Te=T if ok else k
    return SimResult(x=Xt[:Te],z=Zt[:Te],u=Ut[:Te],alpha=At[:Te],
                     y_ref=y_ref_sig[:Te],viol_x=vx[:Te],viol_u=vu[:Te],
                     n_viol=n_viol,ok=ok)


def run_rtmpc(A, B, A_hat, B_hat, K, C_out,
              Hx, h_x, Hu, h_u, Sigma_w,
              rpi, ocp, y_ref_sig, x0, N, rng_seed=None):
    rng=np.random.RandomState(rng_seed)
    T=len(y_ref_sig); n=A.shape[0]; m=B.shape[1]
    Xt=np.zeros((T,n)); Zt=np.zeros((T,n)); Ut=np.zeros((T,m))
    At=np.ones(T); vx=np.zeros(T,bool); vu=np.zeros(T,bool)
    ok=True
    x_k=x0.copy(); z_k=x0.copy()
    for k in range(T):
        zr_k,vr_k=steady_state(A,B,C_out,float(y_ref_sig[k]))
        e_k=x_k-z_k
        z_opt,v_opt,feas=ocp.solve(z_k,zr_k,vr_k)
        if not feas: ok=False; break
        u_k=v_opt[0]+K@e_k
        Xt[k]=x_k; Zt[k]=z_k; Ut[k]=u_k
        vx[k]=bool(np.any(Hx@x_k>h_x+1e-6))
        vu[k]=bool(np.any(Hu@u_k>h_u+1e-6))
        w_k=rng.multivariate_normal(np.zeros(n),Sigma_w)
        x_k=A_hat@x_k+B_hat@u_k+w_k
        z_k=A@Zt[k]+B@v_opt[0]
    Te=T if ok else k
    return SimResult(x=Xt[:Te],z=Zt[:Te],u=Ut[:Te],alpha=At[:Te],
                     y_ref=y_ref_sig[:Te],viol_x=vx[:Te],viol_u=vu[:Te],
                     n_viol=0,ok=ok)


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE
# ══════════════════════════════════════════════════════════════════════════════

def fig_lane_change(res_p, rpi_p, res_r, rpi_r,
                    vx0, Ts, lane_width=3.5,
                    fname="fig_lane_change_comparison_alt.png"):
    T_p=len(res_p.x); T_r=len(res_r.x)
    x_lon_p=np.arange(T_p)*Ts*vx0
    x_lon_r=np.arange(T_r)*Ts*vx0

    s_p=float(np.sqrt(rpi_p.P_star[0,0]))
    s_r=float(np.sqrt(rpi_r.P_star[0,0]))
    alp_p=np.clip(res_p.alpha,0.,1.); alp_r=np.clip(res_r.alpha,0.,1.)

    fig,ax=plt.subplots(1,1,figsize=(ONE_COL,ONE_COL*0.6),dpi=300)

    # ── Road ─────────────────────────────────────────────────────────────────
    ax.axhspan(-lane_width,lane_width,color="0.999",lw=0.,zorder=0)
    ax.axhline( lane_width,color=C["constr"],lw=1.0,zorder=1)
    ax.axhline(-lane_width,color=C["constr"],lw=1.0,zorder=1)
    ax.axhline(0.,color=C["constr"],lw=0.5,alpha=0.35,ls=":",zorder=1)

    # ── Tube projections ─────────────────────────────────────────────────────
    # RT-MPC: alpha=1 always -> full Omega cross-section visible throughout
    ax.fill_between(x_lon_r,
                    res_r.z[:,0]-s_r,
                    res_r.z[:,0]+s_r,
                    color=C["r"],alpha=0.15,lw=0.,zorder=2,
                    label=rf"RT-MPC tube ($\Omega$)")
    # HT-MPC: alpha_k < 1 -> tube contracts to alpha_k * Omega at steady state
    ax.fill_between(x_lon_p,
                    res_p.z[:,0]-alp_p*s_p,
                    res_p.z[:,0]+alp_p*s_p,
                    color=C["p"],alpha=0.30,lw=0.,zorder=3,
                    label=rf"HT-MPC tube ($\alpha_k\Omega$)")

    # ── Reference ────────────────────────────────────────────────────────────
    ax.step(x_lon_p,res_p.y_ref,color=C["ref"],lw=0.4,ls="-",
            where="post",zorder=6,label="Reference")

    # ── Trajectories ─────────────────────────────────────────────────────────
    ax.plot(x_lon_r,res_r.x[:,0],color=C["r"],lw=0.6,ls="--",
            zorder=4,label=LABELS["r"])
    ax.plot(x_lon_p,res_p.x[:,0],color=C["p"],lw=0.6,ls="--",
            zorder=5,label=LABELS["p"])

    # ── Constraint violation markers ─────────────────────────────────────────
    for key,res,x_lon in [("r",res_r,x_lon_r),("p",res_p,x_lon_p)]:
        vi=np.where(res.viol_x)[0]
        if len(vi)>0:
            ax.scatter(x_lon[vi],res.x[vi,0],c=C[key],s=18,
                       marker="x",zorder=7)

    # ── Axes ─────────────────────────────────────────────────────────────────
    ax.set_ylabel(r"$e_y$ [m]",labelpad=1)
    ax.set_xlabel(r"Longitudinal distance [m]",labelpad=1)
    ax.set_ylim(-lane_width-0.35, lane_width+0.35)
    ax.set_xlim([0., max(x_lon_p[-1],x_lon_r[-1])])
    ax.set_yticks([-lane_width/2, 0., lane_width/2])
    ax.set_yticklabels([f"{-lane_width/2:.2f}", "0", f"{lane_width/2:.2f}"])

    # lane labels
    ax.text(2., -lane_width+0.15, "slow lane",
            color=C["constr"],fontsize=FP-2,va="bottom")
    ax.text(2.,  lane_width-0.15, "fast lane",
            color=C["constr"],fontsize=FP-2,va="top")

    # ── Inset zoom (steady-state cruise) ─────────────────────────────────────
    axins=inset_axes(ax,width=1.,height=0.325,
                     bbox_to_anchor=(42.5,-0.1),
                     bbox_transform=ax.transData,
                     loc="upper left",borderpad=0.)
    for sp in axins.spines.values():
        sp.set_linewidth(0.4); sp.set_color("0.4"); sp.set_linestyle("-.")
    axins.fill_between(x_lon_r,res_r.z[:,0]-s_r,res_r.z[:,0]+s_r,
                       color=C["r"],alpha=0.15,lw=0.)
    axins.fill_between(x_lon_p,res_p.z[:,0]-alp_p*s_p,res_p.z[:,0]+alp_p*s_p,
                       color=C["p"],alpha=0.30,lw=0.)
    axins.step(x_lon_p,res_p.y_ref,color=C["ref"],lw=0.4,ls="-",where="post")
    axins.plot(x_lon_r,res_r.x[:,0],color=C["r"],lw=0.6,ls="--")
    axins.plot(x_lon_p,res_p.x[:,0],color=C["p"],lw=0.6,ls="--")
    axins.set_xlim(52.5,72.5); axins.set_ylim(1.25,2.5)
    axins.set_xticks([]); axins.set_yticks([])
    pp,p1,p2=mark_inset(ax,axins,loc1=4,loc2=2,
                         ec="0.4",linewidth=0.4,linestyle="-.")
    pp.set_alpha(0.4); pp.set_linewidth(0.4)
    for c in (p1,p2):
        c.set_linewidth(0.4); c.set_linestyle("-."); c.set_alpha(0.4)

    # ── Legend ───────────────────────────────────────────────────────────────
    legend_handles=[
        Line2D([0],[0],color=C["ref"],lw=0.8,ls="-",   label="Reference"),
        Line2D([0],[0],color=C["p"],  lw=0.9,ls="--",  label=LABELS["p"]),
        Line2D([0],[0],color=C["r"],  lw=0.9,ls="--",  label=LABELS["r"]),
        Patch(facecolor=C["p"],alpha=0.30,label=r"$\alpha_k\Omega$ (HT-MPC)"),
        Patch(facecolor=C["r"],alpha=0.15,label=r"$\Omega$ (RT-MPC)"),
        Line2D([0],[0],color=C["constr"],lw=1.0,label="Lane boundary"),
    ]
    ax.legend(handles=legend_handles,loc="upper center",
              bbox_to_anchor=(0.5,1.325),ncol=3,fontsize=FP-1,frameon=False)

    fig.tight_layout()
    fig.savefig(fname,dpi=600,bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved  {fname}")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    SEP = "="*68
    print(SEP)
    print("  Lateral Vehicle Lane Change  —  HT-MPC vs RT-MPC")
    print("  (identical disturbance bound; scale adaptation is the only difference)")
    print(SEP)

    # ── Vehicle model ─────────────────────────────────────────────────────────
    vx0  = 60.0/3.6   # 60 km/h = 16.67 m/s
    Ts   = 0.1        # 100 ms sample time (ZOH)
    T_sim = 50        # 5 s total
    T_lc  = 12        # lane change at t = 1.2 s

    A, B  = discretize(vx0, Ts)
    n, m  = 4, 2
    C_out = np.array([[1.,0.,0.,0.]])   # output = e_y
    print(f"\n  v_x0={vx0:.3f} m/s ({vx0*3.6:.0f} km/h)  "
          f"Ts={Ts} s  T={T_sim}  T_lc={T_lc}  n={n} m={m}")

    # ── Ancillary LQR gain ────────────────────────────────────────────────────
    Q_lqr = np.diag([50.,10.,1.,10.])
    R_lqr = np.diag([1.0,0.1])
    P_lqr = la.solve_discrete_are(A,B,Q_lqr,R_lqr)
    K     = -la.solve(R_lqr+B.T@P_lqr@B, B.T@P_lqr@A)
    A_K   = A+B@K
    rho   = float(np.max(np.abs(la.eigvals(A_K))))
    det   = float(abs(la.det(A_K)))
    print(f"  rho(A_K)={rho:.6f}   |det(A_K)|={det:.6f}")
    assert rho < 1.0, "A_K not Schur-stable."
    assert det > 1e-9, "A_K not invertible — c_S < 1 not guaranteed."

    # ── Uncertainty: Delta_A = Delta_vx * D,  df_DA = 1 ─────────────────────
    sigma_vx = 0.1                     # m/s standard deviation
    D        = dAd_dvx(vx0, Ts)
    d_flat   = D.flatten()
    eps_reg  = (sigma_vx*0.01)**2

    Sigma_DA = sigma_vx**2*np.outer(d_flat,d_flat) + eps_reg*np.eye(n*n)
    Sigma_DA = 0.5*(Sigma_DA+Sigma_DA.T)
    Sigma_DB = (sigma_vx*1e-3)**2*np.eye(n*m)   # B independent of vx
    Sigma_w  = np.diag([1e-4,2e-5,1e-4,5e-5])*2

    # ── Shared confidence level (both controllers use the SAME bound) ─────────
    p_conf = 0.99      # p_A = p_B = p_W = 0.99

    # ── Constraint sets ───────────────────────────────────────────────────────
    # |e_y|<=3 m, |e_psi|<=0.30 rad, |v_y|<=2 m/s, |psi_dot|<=1.5 rad/s
    # |delta_s|<=0.20 rad, |mu_b|<=2 N·m
    x_bounds = np.array([3.0, 0.30, 2.0, 1.5])
    u_bounds = np.array([0.20, 2.0])
    Hx  = np.vstack([np.eye(n),-np.eye(n)])
    h_x = np.tile(x_bounds,2)
    Hu  = np.vstack([np.eye(m),-np.eye(m)])
    h_u = np.tile(u_bounds,2)

    # ── Stage / terminal cost ─────────────────────────────────────────────────
    Q_c   = np.diag([20.,5.,0.5,0.5])
    R_c   = np.diag([0.1,0.01])
    P_inf = la.solve_discrete_are(A,B,Q_c,R_c)

    # ── Offline: SHARED bound, RPI, tightening ────────────────────────────────
    print("\n[Offline] Shared disturbance bound + RPI ellipsoid ...")
    unc = UncertaintyModel(Sigma_DA, Sigma_DB, Sigma_w,
                           p_conf, p_conf, p_conf,
                           n, m, df_DA=1, df_DB=1, label="shared")
    QA, QB = compute_Qmax(unc, Hx, h_x, Hu, h_u)
    rpi    = compute_rpi(A_K, unc, QA, QB, label="shared")
    tight  = compute_tightening(A_K, K, rpi, Hx, h_x, Hu, h_u,
                                P_inf, label="shared")

    print(f"\n  RPI half-width in e_y direction: "
          f"sqrt(P*[0,0]) = {np.sqrt(rpi.P_star[0,0]):.4f} m")
    print(f"  c_S = {rpi.c_S:.6f}  (1-c_S = {1-rpi.c_S:.6f})")

    # ── Compile OCPs (both from the SAME tightening) ─────────────────────────
    N = 25
    print(f"\n[Compile] OCPs  N={N} ...")
    ocp_p = HomotheticTubeMPC_OCP(A,B,Hx,h_x,Hu,h_u,tight,N,Q_c,R_c,P_inf)
    ocp_r = FixedTubeMPC_OCP(     A,B,Hx,h_x,Hu,h_u,tight,N,Q_c,R_c,P_inf)

    # ── True plant (one fixed draw) ───────────────────────────────────────────
    rng_plant = np.random.RandomState(42)
    Delta_vx  = rng_plant.normal(0., sigma_vx)
    A_hat, B_hat = discretize(vx0+Delta_vx, Ts)
    print(f"\n  Delta_vx = {Delta_vx:+.4f} m/s  ({Delta_vx*3.6:+.3f} km/h)")
    print(f"  ||Delta_A||_F = {la.norm(A_hat-A,'fro'):.6f}")

    # ── Lane change reference ─────────────────────────────────────────────────
    lane_width = 3.5
    y_slow = -lane_width/2.   # -1.75 m
    y_fast =  lane_width/2.   # +1.75 m
    y_ref_sig = np.where(np.arange(T_sim)<T_lc, y_slow, y_fast).astype(float)
    x0 = np.array([y_slow,0.,0.,0.])

    # ── Feasibility check ─────────────────────────────────────────────────────
    print("\n[Check] OCP feasibility at k=0 ...")
    zr0,vr0 = steady_state(A,B,C_out,y_slow)
    aseq0   = alpha_seq(0., rpi.c_S, N)
    _,_,fp0 = ocp_p.solve(x0,zr0,vr0,aseq0)
    _,_,fr0 = ocp_r.solve(x0,zr0,vr0)
    print(f"  HT-MPC feasible: {fp0}   RT-MPC feasible: {fr0}")

    # ── Simulation ────────────────────────────────────────────────────────────
    print(f"\n[Simulate] T={T_sim} steps, lane change at k={T_lc} (t={T_lc*Ts:.1f} s) ...")
    res_p = run_htmpc( A,B,A_hat,B_hat,K,C_out,
                       Hx,h_x,Hu,h_u,Sigma_w,
                       rpi,ocp_p,y_ref_sig,x0,N,rng_seed=7)
    res_r = run_rtmpc( A,B,A_hat,B_hat,K,C_out,
                       Hx,h_x,Hu,h_u,Sigma_w,
                       rpi,ocp_r,y_ref_sig,x0,N,rng_seed=7)

    for lbl,res in [("HT-MPC",res_p),("RT-MPC",res_r)]:
        print(f"\n  {lbl}:")
        print(f"    Status           : {'OK' if res.ok else 'FAILED'}")
        print(f"    State violations : {int(res.viol_x.sum())}")
        print(f"    Input violations : {int(res.viol_u.sum())}")
        print(f"    max alpha_k      : {res.alpha.max():.4f}")
        print(f"    mean alpha_k     : {res.alpha.mean():.4f}")
        print(f"    cap active steps : {int((res.alpha>1.+1e-6).sum())}")

    # ── Figure ────────────────────────────────────────────────────────────────
    print("\n[Figure] ...")
    fig_lane_change(res_p, rpi, res_r, rpi,
                    vx0=vx0, Ts=Ts, lane_width=lane_width)
    print("\n" + SEP + "\n  Done.\n" + SEP)

  Lateral Vehicle Lane Change  —  HT-MPC vs RT-MPC
  (identical disturbance bound; scale adaptation is the only difference)

  v_x0=16.667 m/s (60 km/h)  Ts=0.1 s  T=50  T_lc=12  n=4 m=2
  rho(A_K)=0.685282   |det(A_K)|=0.000319

[Offline] Shared disturbance bound + RPI ellipsoid ...
  OK [shared]  p_A=0.99, p_B=0.99, p_W=0.99  chi2_DA=6.635  chi2_w=13.277
  OK  tr(Q^A_max)=0.00033  tr(Q^B_max)=0.00000
  OK [shared]  lam*=0.7061  c_S=0.999999  tr(P*)=1.1608  sqrt(P*[0,0])=0.3490 m
  OK [shared]  s_x=[0.349  0.1236 0.6118 0.8059 0.349  0.1236 0.6118 0.8059]  gamma=0.04333  margins: [2.651  0.1764 1.3882 0.6941 2.651  0.1764 1.3882 0.6941]

  RPI half-width in e_y direction: sqrt(P*[0,0]) = 0.3490 m
  c_S = 0.999999  (1-c_S = 0.000001)

[Compile] OCPs  N=25 ...
[Compile] HT-MPC OCP ...
  OK  N=25  solver=CLARABEL  terminal_SOC=True
[Compile] RT-MPC OCP ...
  OK  fixed tightening h_x-s_x=[2.651  0.1764 1.3882 0.6941]

  Delta_vx = +0.0497 m/s  (+0.179 km/h)
  ||Delta_A||_F = 0.006020

[Ch

C:\Users\milad.banitalebi\AppData\Local\Temp\ipykernel_18560\2701269597.py:589: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  Saved  fig_lane_change_comparison_alt.png

  Done.
